In [1]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

In [2]:

# =========================
# 0. Config
# =========================

PROJECT_ROOT = Path("/home/zilinm2/futures_anomaly_project")

NB01_DIR = PROJECT_ROOT / "notebook01_base_clean_panel"
NB04_DIR = PROJECT_ROOT / "notebook04_post_selection_decay"
FEATURE_DIR = NB04_DIR / "features"
STRATEGY_DIR = NB04_DIR / "strategy_returns"
TABLE_DIR = NB04_DIR / "tables"
DIAG_DIR = NB04_DIR / "diagnostics"

for d in [NB04_DIR, FEATURE_DIR, STRATEGY_DIR, TABLE_DIR, DIAG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLEAN_PANEL_PARQUET = NB01_DIR / "notebook01_base_clean_panel.parquet"
CLEAN_PANEL_CSV = NB01_DIR / "notebook01_base_clean_panel.csv"

TRADING_DAYS_PER_YEAR = 252
COST_BPS = 5.0
MIN_SYMBOLS_PER_DATE = 6
MIN_IS_OBS = 500
TOP_N_SELECTED = 10
SAVE_SYMBOL_RETURNS_CSV = False

IS_START = pd.Timestamp("2016-01-05")
IS_END = pd.Timestamp("2020-12-31")
OOS_START = pd.Timestamp("2021-01-01")
OOS_END = pd.Timestamp("2023-12-31")
LIVE_START = pd.Timestamp("2024-01-01")

MA_WINDOWS = [10, 20, 60, 120]
TSMOM_WINDOWS = [20, 60, 120]
BREAKOUT_WINDOWS = [20, 60, 120]
REVERSAL_WINDOWS = [3, 5, 10]
VERSIONS = ["long_flat", "long_short"]

RETURN_COLS = [
    "gross_strategy_return",
    "net_strategy_return",
    "passive_return",
    "gross_map_vs_passive",
    "net_map_vs_passive",
    "net_map_vs_zero",
]

CORE_RETURN_COLS = [
    "net_strategy_return",
    "net_map_vs_passive",
    "net_map_vs_zero",
]

RUN_CONFIG = {
    "notebook": "notebook04_post_selection_decay",
    "version": "v7_delayed_return_window_audited",
    "input_clean_panel_parquet": str(CLEAN_PANEL_PARQUET),
    "input_clean_panel_csv": str(CLEAN_PANEL_CSV),
    "strategy_families": ["MA", "TSMOM", "BREAKOUT", "REVERSAL"],
    "ma_windows": MA_WINDOWS,
    "tsmom_windows": TSMOM_WINDOWS,
    "breakout_windows": BREAKOUT_WINDOWS,
    "reversal_windows": REVERSAL_WINDOWS,
    "versions": VERSIONS,
    "cost_bps_per_one_way_notional_turnover": COST_BPS,
    "execution_timing": "strict delayed: signal[t-1] becomes exec_position[t]; return window is close[t] to close[t+1]; IS/OOS splits exclude cross-boundary return windows",
    "is_period": [str(IS_START.date()), str(IS_END.date())],
    "oos_period": [str(OOS_START.date()), str(OOS_END.date())],
    "pseudo_live_start": str(LIVE_START.date()),
    "min_symbols_per_date": MIN_SYMBOLS_PER_DATE,
    "min_is_obs": MIN_IS_OBS,
    "top_n_selected": TOP_N_SELECTED,
    "save_symbol_returns_csv": SAVE_SYMBOL_RETURNS_CSV,
    "annualization": TRADING_DAYS_PER_YEAR,
}

print(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))

{
  "notebook": "notebook04_post_selection_decay",
  "version": "v7_delayed_return_window_audited",
  "input_clean_panel_parquet": "/home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.parquet",
  "input_clean_panel_csv": "/home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.csv",
  "strategy_families": [
    "MA",
    "TSMOM",
    "BREAKOUT",
    "REVERSAL"
  ],
  "ma_windows": [
    10,
    20,
    60,
    120
  ],
  "tsmom_windows": [
    20,
    60,
    120
  ],
  "breakout_windows": [
    20,
    60,
    120
  ],
  "reversal_windows": [
    3,
    5,
    10
  ],
  "versions": [
    "long_flat",
    "long_short"
  ],
  "cost_bps_per_one_way_notional_turnover": 5.0,
  "execution_timing": "strict delayed: signal[t-1] becomes exec_position[t]; return window is close[t] to close[t+1]; IS/OOS splits exclude cross-boundary return windows",
  "is_period": [
    "2016-01-05",
    "2020-12-31"
  ],
  "oos_peri

In [3]:

# =========================
# 1. Helper functions
# =========================

def read_table(parquet_path: Path, csv_path: Path, date_cols=None) -> pd.DataFrame:
    date_cols = date_cols or []
    if parquet_path.exists():
        print(f"Loading parquet: {parquet_path}")
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        print(f"Loading csv: {csv_path}")
        return pd.read_csv(csv_path, parse_dates=date_cols, low_memory=False)
    raise FileNotFoundError(
        "No input table found. Checked:\n"
        f"{parquet_path}\n{csv_path}"
    )


def safe_to_parquet(df: pd.DataFrame, path: Path) -> bool:
    try:
        df.to_parquet(path, index=False)
        return True
    except Exception as exc:
        print(f"[WARN] parquet save failed: {path}")
        print(f"[WARN] {type(exc).__name__}: {exc}")
        return False


def save_table(df: pd.DataFrame, out_dir: Path, name: str, save_csv: bool = True, save_parquet: bool = True) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / f"{name}.csv"
    parquet_path = out_dir / f"{name}.parquet"

    csv_out = None
    parquet_out = None

    if save_csv:
        df.to_csv(csv_path, index=False)
        csv_out = str(csv_path)

    if save_parquet:
        parquet_ok = safe_to_parquet(df, parquet_path)
        parquet_out = str(parquet_path) if parquet_ok else None
        if (not parquet_ok) and (not save_csv):
            # Large but safer than producing no downstream file when parquet engines are unavailable.
            df.to_csv(csv_path, index=False)
            csv_out = str(csv_path)

    return {
        "name": name,
        "rows": int(len(df)),
        "cols": int(df.shape[1]),
        "csv": csv_out,
        "parquet": parquet_out,
    }


def require_columns(df: pd.DataFrame, required_cols, table_name: str):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{table_name} missing required columns: {missing}")


def parse_bool_flag(s: pd.Series, name: str) -> pd.Series:
    """
    Convert input flag columns to strict booleans.

    This avoids pandas treating non-empty strings such as "False" or "0" as True.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)

    if pd.api.types.is_numeric_dtype(s):
        valid = s.dropna()
        bad = ~valid.isin([0, 1])
        if bad.any():
            raise ValueError(
                f"Invalid numeric boolean flag values in {name}: "
                f"{sorted(valid[bad].unique())[:10]}"
            )
        return s.fillna(0).astype(int).astype(bool)

    normalized = s.astype("string").str.strip().str.lower()
    normalized = normalized.mask(normalized == "")
    mapped = normalized.map({
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
        "y": True, "n": False,
    })
    bad = mapped.isna() & normalized.notna()
    if bad.any():
        raise ValueError(
            f"Invalid boolean flag values in {name}: "
            f"{sorted(s[bad].dropna().astype(str).unique())[:10]}"
        )
    return mapped.fillna(False).astype(bool)


def annualized_perf(x: pd.Series, trading_days: int = 252) -> dict:
    x = pd.to_numeric(x, errors="coerce").dropna()
    n = int(len(x))
    if n == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "mean_ann": np.nan,
            "vol_daily": np.nan,
            "vol_ann": np.nan,
            "sharpe": np.nan,
            "t_stat": np.nan,
            "hit_rate": np.nan,
            "min_daily": np.nan,
            "max_daily": np.nan,
            "max_drawdown": np.nan,
        }

    mean_daily = float(x.mean())
    vol_daily = float(x.std(ddof=1)) if n > 1 else np.nan
    mean_ann = mean_daily * trading_days
    vol_ann = vol_daily * math.sqrt(trading_days) if pd.notna(vol_daily) else np.nan
    sharpe = mean_ann / vol_ann if vol_ann and vol_ann > 0 else np.nan
    t_stat = mean_daily / (vol_daily / math.sqrt(n)) if vol_daily and vol_daily > 0 else np.nan

    equity = (1.0 + x).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    max_dd = float(drawdown.min()) if len(drawdown) else np.nan

    return {
        "n_obs": n,
        "mean_daily": mean_daily,
        "mean_ann": float(mean_ann),
        "vol_daily": vol_daily,
        "vol_ann": vol_ann,
        "sharpe": float(sharpe) if pd.notna(sharpe) else np.nan,
        "t_stat": float(t_stat) if pd.notna(t_stat) else np.nan,
        "hit_rate": float((x > 0).mean()),
        "min_daily": float(x.min()),
        "max_daily": float(x.max()),
        "max_drawdown": max_dd,
    }


def add_return_window_split_label(
    df: pd.DataFrame,
    start_col: str = "date",
    end_col: str = "return_end_date",
) -> pd.DataFrame:
    """Assign sample split using the full realized return window.

    A row belongs to IS/OOS only if both return start and return end are inside
    the same split. This prevents the final IS day from using a return that ends
    in OOS, and prevents the final OOS day from using a return that ends in
    pseudo-live. Pseudo-live has no fixed upper bound.
    """
    out = df.copy()
    start = pd.to_datetime(out[start_col])
    end = pd.to_datetime(out[end_col])

    is_mask = (start >= IS_START) & (end <= IS_END)
    oos_mask = (start >= OOS_START) & (end <= OOS_END)
    live_mask = start >= LIVE_START

    out["sample_split"] = np.select(
        [is_mask, oos_mask, live_mask],
        ["IS", "OOS", "PSEUDO_LIVE"],
        default="OUT_OF_SCOPE",
    )
    return out


def summarize_by_group(df: pd.DataFrame, group_cols, return_cols) -> pd.DataFrame:
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False, observed=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_dict = dict(zip(group_cols, keys))
        for col in return_cols:
            perf = annualized_perf(g[col], TRADING_DAYS_PER_YEAR)
            row = {**key_dict, "return_col": col, **perf}
            if "turnover" in g.columns:
                row["avg_turnover_daily"] = float(pd.to_numeric(g["turnover"], errors="coerce").mean())
            if "n_symbols" in g.columns:
                row["avg_n_symbols"] = float(pd.to_numeric(g["n_symbols"], errors="coerce").mean())
            rows.append(row)
    return pd.DataFrame(rows)


def make_strategy_id(strategy_family: str, lookback: int, version: str) -> str:
    return f"{strategy_family}_{lookback}_{version}"


def classify_decay(row: pd.Series) -> str:
    is_mean = row.get("IS_mean_ann", np.nan)
    oos_mean = row.get("OOS_mean_ann", np.nan)
    live_mean = row.get("PSEUDO_LIVE_mean_ann", np.nan)
    is_sharpe = row.get("IS_sharpe", np.nan)
    oos_sharpe = row.get("OOS_sharpe", np.nan)
    live_sharpe = row.get("PSEUDO_LIVE_sharpe", np.nan)

    if pd.isna(is_mean) or pd.isna(oos_mean) or pd.isna(live_mean):
        return "INSUFFICIENT_SPLIT_DATA"
    if is_mean <= 0 or pd.isna(is_sharpe) or is_sharpe <= 0:
        return "NO_POSITIVE_IS_EDGE"
    if oos_mean < 0 or live_mean < 0:
        return "DECAYED_TO_NEGATIVE"

    oos_retention = oos_mean / is_mean if is_mean > 0 else np.nan
    live_retention = live_mean / is_mean if is_mean > 0 else np.nan
    oos_sharpe_retention = oos_sharpe / is_sharpe if is_sharpe > 0 else np.nan
    live_sharpe_retention = live_sharpe / is_sharpe if is_sharpe > 0 else np.nan

    if abs(oos_mean) < 0.01 and abs(live_mean) < 0.01:
        return "DECAYED_TO_ZERO"
    if min(oos_retention, live_retention) < 0.25:
        return "SEVERE_DECAY"
    if min(oos_retention, live_retention) < 0.70 or min(oos_sharpe_retention, live_sharpe_retention) < 0.50:
        return "PARTIAL_DECAY"
    return "ROBUST"


In [4]:

# =========================
# 2. Load Notebook 01 clean panel
# =========================

panel = read_table(CLEAN_PANEL_PARQUET, CLEAN_PANEL_CSV, date_cols=["date", "trading_day", "datetime"])

if "date" not in panel.columns and "trading_day" in panel.columns:
    panel["date"] = pd.to_datetime(panel["trading_day"])
else:
    panel["date"] = pd.to_datetime(panel["date"])

required_panel_cols = [
    "date", "symbol", "open", "high", "low", "close",
    "fwd_ret_1d", "fwd_tradable_return_flag",
]
require_columns(panel, required_panel_cols, "clean panel")

# Normalize tradability flag once before any filtering. Avoid pandas string-to-bool pitfalls.
panel["fwd_tradable_return_flag"] = parse_bool_flag(panel["fwd_tradable_return_flag"], "fwd_tradable_return_flag")
assert pd.api.types.is_bool_dtype(panel["fwd_tradable_return_flag"]), "fwd_tradable_return_flag was not converted to bool."

panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)

if panel.duplicated(["symbol", "date"]).any():
    dup_n = int(panel.duplicated(["symbol", "date"]).sum())
    raise ValueError(f"Duplicate symbol-date rows in clean panel: {dup_n}")

panel["prev_date"] = panel.groupby("symbol")["date"].shift(1)
panel["next_date"] = panel.groupby("symbol")["date"].shift(-1)

print("panel shape:", panel.shape)
print("symbols:", panel["symbol"].nunique())
print("date range:", panel["date"].min(), "->", panel["date"].max())
print(panel[["date", "symbol", "close", "fwd_ret_1d"]].head(5).to_string(index=False))


Loading parquet: /home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.parquet
panel shape: (58591, 69)
symbols: 27
date range: 2016-01-05 00:00:00 -> 2026-06-18 00:00:00
      date symbol  close  fwd_ret_1d
2016-01-05     AG 3313.0    0.000302
2016-01-06     AG 3314.0    0.012975
2016-01-07     AG 3357.0   -0.000298
2016-01-08     AG 3356.0   -0.002086
2016-01-11     AG 3349.0   -0.018214


In [5]:

# =========================
# 3. Build signal features
# =========================

feature_panel = panel[[
    "date", "prev_date", "next_date", "symbol", "open", "high", "low", "close",
    "fwd_ret_1d", "fwd_tradable_return_flag"
]].copy()

feature_panel["passive_return"] = feature_panel["fwd_ret_1d"]

all_windows = sorted(set(MA_WINDOWS + TSMOM_WINDOWS + BREAKOUT_WINDOWS + REVERSAL_WINDOWS))
g = feature_panel.groupby("symbol", group_keys=False)

for w in sorted(set(MA_WINDOWS)):
    feature_panel[f"ma_{w}"] = g["close"].transform(lambda s, w=w: s.rolling(w, min_periods=w).mean())

for w in sorted(set(TSMOM_WINDOWS + REVERSAL_WINDOWS)):
    feature_panel[f"mom_{w}"] = g["close"].transform(lambda s, w=w: s.pct_change(w))

for w in sorted(set(BREAKOUT_WINDOWS)):
    feature_panel[f"prior_high_{w}"] = g["high"].transform(lambda s, w=w: s.shift(1).rolling(w, min_periods=w).max())
    feature_panel[f"prior_low_{w}"] = g["low"].transform(lambda s, w=w: s.shift(1).rolling(w, min_periods=w).min())

print("feature_panel shape:", feature_panel.shape)
print(feature_panel[["date", "symbol", "close", "fwd_ret_1d"]].head(5).to_string(index=False))

feature_panel shape: (58591, 27)
      date symbol  close  fwd_ret_1d
2016-01-05     AG 3313.0    0.000302
2016-01-06     AG 3314.0    0.012975
2016-01-07     AG 3357.0   -0.000298
2016-01-08     AG 3356.0   -0.002086
2016-01-11     AG 3349.0   -0.018214


In [6]:

# =========================
# 4. Strategy construction
# =========================

def position_from_signal_value(signal_value: pd.Series, version: str, family: str) -> pd.Series:
    x = pd.to_numeric(signal_value, errors="coerce")
    out = pd.Series(np.nan, index=x.index, dtype="float64")

    if family in ["MA", "TSMOM"]:
        if version == "long_flat":
            out = pd.Series(np.where(x.notna(), np.where(x > 0, 1.0, 0.0), np.nan), index=x.index)
        elif version == "long_short":
            out = pd.Series(np.where(x.notna(), np.where(x > 0, 1.0, -1.0), np.nan), index=x.index)
        else:
            raise ValueError(f"Unknown version: {version}")

    elif family == "REVERSAL":
        # Short-term reversal: go long after negative recent return; short/flat after positive recent return.
        if version == "long_flat":
            out = pd.Series(np.where(x.notna(), np.where(x < 0, 1.0, 0.0), np.nan), index=x.index)
        elif version == "long_short":
            out = pd.Series(np.where(x.notna(), np.where(x < 0, 1.0, -1.0), np.nan), index=x.index)
        else:
            raise ValueError(f"Unknown version: {version}")

    else:
        raise ValueError(f"Unsupported family for vector signal: {family}")

    return out.astype("float64")


def breakout_state_for_symbol(close: pd.Series, upper: pd.Series, lower: pd.Series, version: str) -> pd.Series:
    state = np.full(len(close), np.nan, dtype="float64")
    current = 0.0 if version == "long_flat" else np.nan

    c = close.to_numpy(dtype="float64")
    u = upper.to_numpy(dtype="float64")
    l = lower.to_numpy(dtype="float64")

    for i in range(len(c)):
        if np.isnan(c[i]) or np.isnan(u[i]) or np.isnan(l[i]):
            state[i] = np.nan
            continue

        if c[i] > u[i]:
            current = 1.0
        elif c[i] < l[i]:
            current = 0.0 if version == "long_flat" else -1.0

        state[i] = current

    return pd.Series(state, index=close.index)


def add_delayed_execution_and_returns(signal_df: pd.DataFrame, strategy_family: str, lookback: int, version: str) -> pd.DataFrame:
    out = signal_df[[
        "date", "prev_date", "next_date", "symbol", "close", "fwd_ret_1d", "fwd_tradable_return_flag",
        "passive_return", "signal_position", "signal_value"
    ]].copy()

    out["strategy_family"] = strategy_family
    out["lookback"] = int(lookback)
    out["version"] = version
    out["strategy_id"] = make_strategy_id(strategy_family, int(lookback), version)

    by = out.groupby("symbol", group_keys=False)
    out["signal_date"] = by["date"].shift(1)
    out["signal_close"] = by["close"].shift(1)
    out["exec_position"] = by["signal_position"].shift(1)
    out["exec_signal_value"] = by["signal_value"].shift(1)
    out["return_start_date"] = out["date"]
    out["return_end_date"] = out["next_date"]

    out = out[
        out["fwd_tradable_return_flag"]
        & out["exec_position"].notna()
        & out["signal_date"].notna()
        & out["return_end_date"].notna()
    ].copy()

    by2 = out.groupby(["symbol", "strategy_id"], group_keys=False)
    out["prev_exec_position"] = by2["exec_position"].shift(1).fillna(0.0)
    out["turnover"] = (out["exec_position"] - out["prev_exec_position"]).abs()

    out["gross_strategy_return"] = out["exec_position"] * out["fwd_ret_1d"]
    out["cost_return"] = out["turnover"] * COST_BPS / 10000.0
    out["net_strategy_return"] = out["gross_strategy_return"] - out["cost_return"]
    out["gross_map_vs_passive"] = out["gross_strategy_return"] - out["passive_return"]
    out["net_map_vs_passive"] = out["net_strategy_return"] - out["passive_return"]
    out["net_map_vs_zero"] = out["net_strategy_return"]

    return out


def build_ma_strategy(feature_panel: pd.DataFrame, window: int, version: str) -> pd.DataFrame:
    x = feature_panel.copy()
    x["signal_value"] = x["close"] / x[f"ma_{window}"] - 1.0
    x["signal_position"] = position_from_signal_value(x["signal_value"], version, "MA")
    return add_delayed_execution_and_returns(x, "MA", window, version)


def build_tsmom_strategy(feature_panel: pd.DataFrame, window: int, version: str) -> pd.DataFrame:
    x = feature_panel.copy()
    x["signal_value"] = x[f"mom_{window}"]
    x["signal_position"] = position_from_signal_value(x["signal_value"], version, "TSMOM")
    return add_delayed_execution_and_returns(x, "TSMOM", window, version)


def build_reversal_strategy(feature_panel: pd.DataFrame, window: int, version: str) -> pd.DataFrame:
    x = feature_panel.copy()
    x["signal_value"] = x[f"mom_{window}"]
    x["signal_position"] = position_from_signal_value(x["signal_value"], version, "REVERSAL")
    return add_delayed_execution_and_returns(x, "REVERSAL", window, version)


def build_breakout_strategy(feature_panel: pd.DataFrame, window: int, version: str) -> pd.DataFrame:
    x = feature_panel.copy()
    upper_col = f"prior_high_{window}"
    lower_col = f"prior_low_{window}"

    x["signal_value"] = np.nan
    upper_break = x["close"] / x[upper_col] - 1.0
    lower_break = x["close"] / x[lower_col] - 1.0
    x.loc[x[upper_col].notna(), "signal_value"] = upper_break
    x.loc[x[lower_col].notna() & (x["close"] < x[lower_col]), "signal_value"] = lower_break

    state_parts = []
    for _, gsym in x.groupby("symbol", sort=False):
        s = breakout_state_for_symbol(gsym["close"], gsym[upper_col], gsym[lower_col], version)
        state_parts.append(s)
    x["signal_position"] = pd.concat(state_parts).sort_index()

    return add_delayed_execution_and_returns(x, "BREAKOUT", window, version)


strategy_specs = []
for w in MA_WINDOWS:
    for v in VERSIONS:
        strategy_specs.append(("MA", w, v))
for w in TSMOM_WINDOWS:
    for v in VERSIONS:
        strategy_specs.append(("TSMOM", w, v))
for w in BREAKOUT_WINDOWS:
    for v in VERSIONS:
        strategy_specs.append(("BREAKOUT", w, v))
for w in REVERSAL_WINDOWS:
    for v in VERSIONS:
        strategy_specs.append(("REVERSAL", w, v))

strategy_pool = pd.DataFrame(strategy_specs, columns=["strategy_family", "lookback", "version"])
strategy_pool["strategy_id"] = strategy_pool.apply(
    lambda r: make_strategy_id(r["strategy_family"], int(r["lookback"]), r["version"]), axis=1
)

print("candidate strategy count:", len(strategy_pool))
print(strategy_pool.to_string(index=False))


candidate strategy count: 26
strategy_family  lookback    version             strategy_id
             MA        10  long_flat         MA_10_long_flat
             MA        10 long_short        MA_10_long_short
             MA        20  long_flat         MA_20_long_flat
             MA        20 long_short        MA_20_long_short
             MA        60  long_flat         MA_60_long_flat
             MA        60 long_short        MA_60_long_short
             MA       120  long_flat        MA_120_long_flat
             MA       120 long_short       MA_120_long_short
          TSMOM        20  long_flat      TSMOM_20_long_flat
          TSMOM        20 long_short     TSMOM_20_long_short
          TSMOM        60  long_flat      TSMOM_60_long_flat
          TSMOM        60 long_short     TSMOM_60_long_short
          TSMOM       120  long_flat     TSMOM_120_long_flat
          TSMOM       120 long_short    TSMOM_120_long_short
       BREAKOUT        20  long_flat   BREAKOUT_20_long_

In [7]:

# =========================
# 5. Generate candidate strategy returns
# =========================

strategy_frames = []

for family, lookback, version in strategy_specs:
    print(f"Building {family} {lookback} {version} ...")
    if family == "MA":
        tmp = build_ma_strategy(feature_panel, lookback, version)
    elif family == "TSMOM":
        tmp = build_tsmom_strategy(feature_panel, lookback, version)
    elif family == "BREAKOUT":
        tmp = build_breakout_strategy(feature_panel, lookback, version)
    elif family == "REVERSAL":
        tmp = build_reversal_strategy(feature_panel, lookback, version)
    else:
        raise ValueError(f"Unknown family: {family}")
    strategy_frames.append(tmp)

if not strategy_frames:
    raise ValueError("No strategy frames were generated. Check strategy_specs.")
if not any(len(x) > 0 for x in strategy_frames):
    raise ValueError("All generated strategy frames are empty. Check signal availability and tradability flags.")

candidate_returns = pd.concat(strategy_frames, ignore_index=True)
candidate_returns = candidate_returns.sort_values(["strategy_id", "symbol", "date"]).reset_index(drop=True)

candidate_returns = add_return_window_split_label(candidate_returns, "date", "return_end_date")
candidate_returns = candidate_returns[candidate_returns["sample_split"] != "OUT_OF_SCOPE"].copy()

print("candidate_returns shape:", candidate_returns.shape)
print("strategies:", candidate_returns["strategy_id"].nunique())
print("date range:", candidate_returns["date"].min(), "->", candidate_returns["date"].max())
print(candidate_returns[["date", "signal_date", "return_end_date", "symbol", "strategy_id", "exec_position", "fwd_ret_1d", "net_strategy_return", "sample_split"]].head(10).to_string(index=False))


Building MA 10 long_flat ...
Building MA 10 long_short ...
Building MA 20 long_flat ...
Building MA 20 long_short ...
Building MA 60 long_flat ...
Building MA 60 long_short ...
Building MA 120 long_flat ...
Building MA 120 long_short ...
Building TSMOM 20 long_flat ...
Building TSMOM 20 long_short ...
Building TSMOM 60 long_flat ...
Building TSMOM 60 long_short ...
Building TSMOM 120 long_flat ...
Building TSMOM 120 long_short ...
Building BREAKOUT 20 long_flat ...
Building BREAKOUT 20 long_short ...
Building BREAKOUT 60 long_flat ...
Building BREAKOUT 60 long_short ...
Building BREAKOUT 120 long_flat ...
Building BREAKOUT 120 long_short ...
Building REVERSAL 3 long_flat ...
Building REVERSAL 3 long_short ...
Building REVERSAL 5 long_flat ...
Building REVERSAL 5 long_short ...
Building REVERSAL 10 long_flat ...
Building REVERSAL 10 long_short ...
candidate_returns shape: (1484852, 29)
strategies: 26
date range: 2016-01-11 00:00:00 -> 2026-06-17 00:00:00
      date signal_date return_en

In [8]:

# =========================
# 6. Equal-weight portfolio returns by strategy_id
# =========================

portfolio_group_cols = ["date", "strategy_id", "strategy_family", "lookback", "version", "sample_split"]

portfolio_daily = (
    candidate_returns
    .groupby(portfolio_group_cols, dropna=False, observed=True)
    .agg(
        n_symbols=("symbol", "nunique"),
        turnover=("turnover", "mean"),
        gross_strategy_return=("gross_strategy_return", "mean"),
        net_strategy_return=("net_strategy_return", "mean"),
        passive_return=("passive_return", "mean"),
        gross_map_vs_passive=("gross_map_vs_passive", "mean"),
        net_map_vs_passive=("net_map_vs_passive", "mean"),
        net_map_vs_zero=("net_map_vs_zero", "mean"),
    )
    .reset_index()
)

portfolio_daily = portfolio_daily[portfolio_daily["n_symbols"] >= MIN_SYMBOLS_PER_DATE].copy()
portfolio_daily = portfolio_daily.sort_values(["strategy_id", "date"]).reset_index(drop=True)

print("portfolio_daily shape:", portfolio_daily.shape)
print("portfolio strategies:", portfolio_daily["strategy_id"].nunique())
print(portfolio_daily[["date", "strategy_id", "sample_split", "n_symbols", "turnover", "net_strategy_return"]].head(10).to_string(index=False))

portfolio_daily shape: (64626, 14)
portfolio strategies: 26
      date            strategy_id sample_split  n_symbols  turnover  net_strategy_return
2016-07-05 BREAKOUT_120_long_flat           IS         20       0.3            -0.002170
2016-07-06 BREAKOUT_120_long_flat           IS         20       0.0            -0.001898
2016-07-07 BREAKOUT_120_long_flat           IS         20       0.0            -0.005163
2016-07-08 BREAKOUT_120_long_flat           IS         20       0.0             0.006807
2016-07-11 BREAKOUT_120_long_flat           IS         20       0.0             0.002633
2016-07-12 BREAKOUT_120_long_flat           IS         20       0.0             0.002294
2016-07-13 BREAKOUT_120_long_flat           IS         20       0.0            -0.000335
2016-07-14 BREAKOUT_120_long_flat           IS         20       0.0             0.000884
2016-07-15 BREAKOUT_120_long_flat           IS         20       0.0            -0.003157
2016-07-18 BREAKOUT_120_long_flat           IS    

In [9]:

# =========================
# 7. Split performance and selection scores
# =========================

strategy_perf_by_split = summarize_by_group(
    portfolio_daily,
    ["strategy_id", "strategy_family", "lookback", "version", "sample_split"],
    RETURN_COLS,
)

# Main selection uses net strategy return, in-sample only.
is_selection = strategy_perf_by_split[
    (strategy_perf_by_split["sample_split"] == "IS")
    & (strategy_perf_by_split["return_col"] == "net_strategy_return")
].copy()

is_selection["eligible_for_selection"] = (
    (is_selection["n_obs"] >= MIN_IS_OBS)
    & is_selection["sharpe"].notna()
    & (is_selection["vol_ann"] > 0)
)

# Score deliberately penalizes turnover and drawdown; it is not pure IS Sharpe.
is_selection["selection_score"] = (
    is_selection["sharpe"].fillna(-999.0)
    - 0.25 * is_selection["avg_turnover_daily"].fillna(0.0)
    - 0.10 * is_selection["max_drawdown"].abs().fillna(0.0)
)

selected_strategies = (
    is_selection[is_selection["eligible_for_selection"]]
    .sort_values("selection_score", ascending=False)
    .head(TOP_N_SELECTED)
    .reset_index(drop=True)
)

selected_ids = selected_strategies["strategy_id"].tolist()

print("strategy_perf_by_split shape:", strategy_perf_by_split.shape)
print("selected strategy count:", len(selected_strategies))
print(selected_strategies[["strategy_id", "strategy_family", "lookback", "version", "mean_ann", "sharpe", "t_stat", "max_drawdown", "avg_turnover_daily", "selection_score"]].to_string(index=False))

strategy_perf_by_split shape: (468, 19)
selected strategy count: 10
           strategy_id strategy_family  lookback    version  mean_ann   sharpe   t_stat  max_drawdown  avg_turnover_daily  selection_score
 BREAKOUT_20_long_flat        BREAKOUT        20  long_flat  0.089346 1.012624 2.205119     -0.121995            0.022041         0.994914
    TSMOM_20_long_flat           TSMOM        20  long_flat  0.087033 1.000947 2.179690     -0.111055            0.098950         0.965104
       MA_20_long_flat              MA        20  long_flat  0.070643 0.840590 1.831259     -0.136791            0.117094         0.797638
       MA_60_long_flat              MA        60  long_flat  0.063659 0.709978 1.520629     -0.147362            0.061708         0.679814
       MA_10_long_flat              MA        10  long_flat  0.057409 0.702754 1.537364     -0.140661            0.170748         0.646001
BREAKOUT_20_long_short        BREAKOUT        20 long_short  0.059783 0.640006 1.392529     -0.125

In [10]:

# =========================
# 8. Post-selection decay summary
# =========================

def build_decay_summary(perf: pd.DataFrame, strategy_ids=None) -> pd.DataFrame:
    base = perf[perf["return_col"] == "net_strategy_return"].copy()
    if strategy_ids is not None:
        base = base[base["strategy_id"].isin(strategy_ids)].copy()

    metric_cols = [
        "n_obs", "mean_daily", "mean_ann", "vol_ann", "sharpe", "t_stat",
        "hit_rate", "max_drawdown", "avg_turnover_daily", "avg_n_symbols"
    ]

    wide = base.pivot_table(
        index=["strategy_id", "strategy_family", "lookback", "version"],
        columns="sample_split",
        values=metric_cols,
        aggfunc="first",
    )

    wide.columns = [f"{split}_{metric}" for metric, split in wide.columns]
    wide = wide.reset_index()

    for split in ["IS", "OOS", "PSEUDO_LIVE"]:
        for metric in metric_cols:
            col = f"{split}_{metric}"
            if col not in wide.columns:
                wide[col] = np.nan

    wide["OOS_decay_ann"] = wide["OOS_mean_ann"] - wide["IS_mean_ann"]
    wide["PSEUDO_LIVE_decay_ann"] = wide["PSEUDO_LIVE_mean_ann"] - wide["IS_mean_ann"]
    wide["OOS_retention_ratio"] = np.where(wide["IS_mean_ann"] > 0, wide["OOS_mean_ann"] / wide["IS_mean_ann"], np.nan)
    wide["PSEUDO_LIVE_retention_ratio"] = np.where(wide["IS_mean_ann"] > 0, wide["PSEUDO_LIVE_mean_ann"] / wide["IS_mean_ann"], np.nan)
    wide["OOS_sharpe_retention"] = np.where(wide["IS_sharpe"] > 0, wide["OOS_sharpe"] / wide["IS_sharpe"], np.nan)
    wide["PSEUDO_LIVE_sharpe_retention"] = np.where(wide["IS_sharpe"] > 0, wide["PSEUDO_LIVE_sharpe"] / wide["IS_sharpe"], np.nan)
    wide["decay_flag"] = wide.apply(classify_decay, axis=1)

    return wide.sort_values(["decay_flag", "IS_sharpe"], ascending=[True, False]).reset_index(drop=True)


all_strategy_decay_summary = build_decay_summary(strategy_perf_by_split, strategy_ids=None)
post_selection_decay_summary = build_decay_summary(strategy_perf_by_split, strategy_ids=selected_ids)

# Merge in selection scores for selected strategies.
selection_cols = [
    "strategy_id", "selection_score", "eligible_for_selection",
    "mean_ann", "sharpe", "t_stat", "max_drawdown", "avg_turnover_daily"
]
selection_score_map = selected_strategies[selection_cols].rename(columns={
    "mean_ann": "selection_IS_mean_ann",
    "sharpe": "selection_IS_sharpe",
    "t_stat": "selection_IS_t_stat",
    "max_drawdown": "selection_IS_max_drawdown",
    "avg_turnover_daily": "selection_IS_avg_turnover_daily",
})
post_selection_decay_summary = post_selection_decay_summary.merge(selection_score_map, on="strategy_id", how="left")
post_selection_decay_summary = post_selection_decay_summary.sort_values("selection_score", ascending=False).reset_index(drop=True)

print("all_strategy_decay_summary shape:", all_strategy_decay_summary.shape)
print("post_selection_decay_summary shape:", post_selection_decay_summary.shape)
print(post_selection_decay_summary[["strategy_id", "strategy_family", "lookback", "version", "selection_score", "IS_mean_ann", "OOS_mean_ann", "PSEUDO_LIVE_mean_ann", "IS_sharpe", "OOS_sharpe", "PSEUDO_LIVE_sharpe", "decay_flag"]].head(20).to_string(index=False))

all_strategy_decay_summary shape: (26, 41)
post_selection_decay_summary shape: (10, 48)
           strategy_id strategy_family  lookback    version  selection_score  IS_mean_ann  OOS_mean_ann  PSEUDO_LIVE_mean_ann  IS_sharpe  OOS_sharpe  PSEUDO_LIVE_sharpe          decay_flag
 BREAKOUT_20_long_flat        BREAKOUT        20  long_flat         0.994914     0.089346      0.049308              0.035490   1.012624    0.491327            0.405426       PARTIAL_DECAY
    TSMOM_20_long_flat           TSMOM        20  long_flat         0.965104     0.087033      0.052853              0.020688   1.000947    0.527363            0.240997        SEVERE_DECAY
       MA_20_long_flat              MA        20  long_flat         0.797638     0.070643      0.049604              0.000065   0.840590    0.495366            0.000762        SEVERE_DECAY
       MA_60_long_flat              MA        60  long_flat         0.679814     0.063659      0.040075              0.028192   0.709978    0.381306        

In [11]:

# =========================
# 9. Diagnostics and hard self-check
# =========================

strategy_pool_summary = (
    candidate_returns
    .groupby(["strategy_id", "strategy_family", "lookback", "version"], observed=True)
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        n_rows=("date", "size"),
        n_symbols=("symbol", "nunique"),
        avg_turnover_daily=("turnover", "mean"),
        avg_abs_position=("exec_position", lambda x: float(np.abs(x).mean())),
    )
    .reset_index()
)

split_coverage = (
    portfolio_daily
    .groupby(["strategy_id", "sample_split"], observed=True)
    .agg(n_obs=("date", "size"), first_date=("date", "min"), last_date=("date", "max"), avg_n_symbols=("n_symbols", "mean"))
    .reset_index()
)

# Hard checks
checks = {}
checks["candidate_duplicate_key_count"] = int(candidate_returns.duplicated(["date", "symbol", "strategy_id"]).sum())
checks["portfolio_duplicate_key_count"] = int(portfolio_daily.duplicated(["date", "strategy_id"]).sum())
checks["perf_duplicate_key_count"] = int(strategy_perf_by_split.duplicated(["strategy_id", "sample_split", "return_col"]).sum())
checks["selected_duplicate_strategy_count"] = int(selected_strategies.duplicated(["strategy_id"]).sum())
checks["decay_duplicate_strategy_count"] = int(post_selection_decay_summary.duplicated(["strategy_id"]).sum())

checks["signal_date_ge_execution_date_count"] = int((pd.to_datetime(candidate_returns["signal_date"]) >= pd.to_datetime(candidate_returns["date"])).sum())
checks["signal_date_not_prev_date_count"] = int((pd.to_datetime(candidate_returns["signal_date"]) != pd.to_datetime(candidate_returns["prev_date"])).sum())
checks["return_end_date_le_start_date_count"] = int((pd.to_datetime(candidate_returns["return_end_date"]) <= pd.to_datetime(candidate_returns["date"])).sum())
checks["return_end_date_not_next_date_count"] = int((pd.to_datetime(candidate_returns["return_end_date"]) != pd.to_datetime(candidate_returns["next_date"])).sum())

is_rows = candidate_returns["sample_split"].eq("IS")
oos_rows = candidate_returns["sample_split"].eq("OOS")
live_rows = candidate_returns["sample_split"].eq("PSEUDO_LIVE")
checks["is_cross_boundary_return_count"] = int((is_rows & (pd.to_datetime(candidate_returns["return_end_date"]) > IS_END)).sum())
checks["oos_cross_boundary_return_count"] = int((oos_rows & (pd.to_datetime(candidate_returns["return_end_date"]) > OOS_END)).sum())
checks["live_before_live_start_count"] = int((live_rows & (pd.to_datetime(candidate_returns["date"]) < LIVE_START)).sum())
checks["out_of_scope_count_after_split_filter"] = int(candidate_returns["sample_split"].eq("OUT_OF_SCOPE").sum())

formula_err = {}
formula_err["gross"] = float((candidate_returns["gross_strategy_return"] - candidate_returns["exec_position"] * candidate_returns["fwd_ret_1d"]).abs().max())
formula_err["net"] = float((candidate_returns["net_strategy_return"] - (candidate_returns["gross_strategy_return"] - candidate_returns["cost_return"])).abs().max())
formula_err["gross_map_vs_passive"] = float((candidate_returns["gross_map_vs_passive"] - (candidate_returns["gross_strategy_return"] - candidate_returns["passive_return"])).abs().max())
formula_err["net_map_vs_passive"] = float((candidate_returns["net_map_vs_passive"] - (candidate_returns["net_strategy_return"] - candidate_returns["passive_return"])).abs().max())
formula_err["net_map_vs_zero"] = float((candidate_returns["net_map_vs_zero"] - candidate_returns["net_strategy_return"]).abs().max())
formula_err["cost"] = float((candidate_returns["cost_return"] - candidate_returns["turnover"] * COST_BPS / 10000.0).abs().max())

checks["negative_turnover_count"] = int((candidate_returns["turnover"] < -1e-15).sum())
checks["negative_cost_count"] = int((candidate_returns["cost_return"] < -1e-15).sum())

long_flat_bad = candidate_returns[
    (candidate_returns["version"] == "long_flat")
    & ~candidate_returns["exec_position"].isin([0.0, 1.0])
]
long_short_bad = candidate_returns[
    (candidate_returns["version"] == "long_short")
    & ~candidate_returns["exec_position"].isin([-1.0, 1.0])
]
checks["bad_long_flat_position_count"] = int(len(long_flat_bad))
checks["bad_long_short_position_count"] = int(len(long_short_bad))

checks["strategy_count"] = int(candidate_returns["strategy_id"].nunique())
checks["expected_strategy_count"] = int(len(strategy_pool))
checks["selected_count"] = int(len(selected_strategies))

if checks["candidate_duplicate_key_count"] != 0:
    raise AssertionError(checks)
if checks["portfolio_duplicate_key_count"] != 0:
    raise AssertionError(checks)
if checks["perf_duplicate_key_count"] != 0:
    raise AssertionError(checks)
if checks["selected_duplicate_strategy_count"] != 0:
    raise AssertionError(checks)
if checks["decay_duplicate_strategy_count"] != 0:
    raise AssertionError(checks)
if checks["signal_date_ge_execution_date_count"] != 0:
    raise AssertionError(checks)
if checks["signal_date_not_prev_date_count"] != 0:
    raise AssertionError(checks)
if checks["return_end_date_le_start_date_count"] != 0:
    raise AssertionError(checks)
if checks["return_end_date_not_next_date_count"] != 0:
    raise AssertionError(checks)
if checks["is_cross_boundary_return_count"] != 0:
    raise AssertionError(checks)
if checks["oos_cross_boundary_return_count"] != 0:
    raise AssertionError(checks)
if checks["live_before_live_start_count"] != 0:
    raise AssertionError(checks)
if checks["out_of_scope_count_after_split_filter"] != 0:
    raise AssertionError(checks)
if checks["negative_turnover_count"] != 0 or checks["negative_cost_count"] != 0:
    raise AssertionError(checks)
if checks["bad_long_flat_position_count"] != 0 or checks["bad_long_short_position_count"] != 0:
    raise AssertionError(checks)
if checks["strategy_count"] != checks["expected_strategy_count"]:
    raise AssertionError(checks)
if checks["selected_count"] == 0:
    raise AssertionError(checks)

for k, v in formula_err.items():
    if not np.isfinite(v) or v > 1e-12:
        raise AssertionError({"formula_err": formula_err})

print("Notebook 04 hard self-check: PASSED")
print(json.dumps(checks, indent=2, default=str))
print("formula errors:", formula_err)

print(strategy_pool_summary.head(10).to_string(index=False))
print(split_coverage.head(10).to_string(index=False))

Notebook 04 hard self-check: PASSED
{
  "candidate_duplicate_key_count": 0,
  "portfolio_duplicate_key_count": 0,
  "perf_duplicate_key_count": 0,
  "selected_duplicate_strategy_count": 0,
  "decay_duplicate_strategy_count": 0,
  "signal_date_ge_execution_date_count": 0,
  "signal_date_not_prev_date_count": 0,
  "return_end_date_le_start_date_count": 0,
  "return_end_date_not_next_date_count": 0,
  "is_cross_boundary_return_count": 0,
  "oos_cross_boundary_return_count": 0,
  "live_before_live_start_count": 0,
  "out_of_scope_count_after_split_filter": 0,
  "negative_turnover_count": 0,
  "negative_cost_count": 0,
  "bad_long_flat_position_count": 0,
  "bad_long_short_position_count": 0,
  "strategy_count": 26,
  "expected_strategy_count": 26,
  "selected_count": 10
}
formula errors: {'gross': 0.0, 'net': 0.0, 'gross_map_vs_passive': 0.0, 'net_map_vs_passive': 0.0, 'net_map_vs_zero': 0.0, 'cost': 0.0}
            strategy_id strategy_family  lookback    version first_date  last_date  n

In [12]:

# =========================
# 10. Save outputs
# =========================

outputs = []

outputs.append(save_table(feature_panel, FEATURE_DIR, "notebook04_feature_panel", save_csv=False, save_parquet=True))
outputs.append(save_table(strategy_pool, TABLE_DIR, "notebook04_strategy_pool", save_csv=True, save_parquet=True))
outputs.append(save_table(candidate_returns, STRATEGY_DIR, "notebook04_symbol_candidate_strategy_returns", save_csv=SAVE_SYMBOL_RETURNS_CSV, save_parquet=True))
outputs.append(save_table(portfolio_daily, STRATEGY_DIR, "notebook04_strategy_portfolio_daily_returns", save_csv=True, save_parquet=True))
outputs.append(save_table(strategy_pool_summary, TABLE_DIR, "notebook04_strategy_pool_summary", save_csv=True, save_parquet=True))
outputs.append(save_table(split_coverage, TABLE_DIR, "notebook04_split_coverage", save_csv=True, save_parquet=True))
outputs.append(save_table(strategy_perf_by_split, TABLE_DIR, "notebook04_strategy_performance_by_split", save_csv=True, save_parquet=True))
outputs.append(save_table(is_selection, TABLE_DIR, "notebook04_in_sample_selection_scores", save_csv=True, save_parquet=True))
outputs.append(save_table(selected_strategies, TABLE_DIR, "notebook04_selected_strategies", save_csv=True, save_parquet=True))
outputs.append(save_table(all_strategy_decay_summary, TABLE_DIR, "notebook04_all_strategy_decay_summary", save_csv=True, save_parquet=True))
outputs.append(save_table(post_selection_decay_summary, TABLE_DIR, "notebook04_post_selection_decay_summary", save_csv=True, save_parquet=True))

with open(NB04_DIR / "notebook04_run_config.json", "w", encoding="utf-8") as f:
    json.dump(RUN_CONFIG, f, indent=2, ensure_ascii=False, default=str)

with open(DIAG_DIR / "notebook04_hard_self_check.json", "w", encoding="utf-8") as f:
    json.dump({"checks": checks, "formula_errors": formula_err}, f, indent=2, ensure_ascii=False, default=str)

print("Saved outputs:")
for item in outputs:
    print(json.dumps(item, indent=2, ensure_ascii=False))

print("\nNotebook 04 finished.")
print("Main downstream input for Notebook 05:")
print(STRATEGY_DIR / "notebook04_symbol_candidate_strategy_returns.parquet")
print("If parquet save fails, a CSV fallback is written automatically:")
print(STRATEGY_DIR / "notebook04_symbol_candidate_strategy_returns.csv")
print("\nMain result table:")
print(TABLE_DIR / "notebook04_post_selection_decay_summary.csv")

Saved outputs:
{
  "name": "notebook04_feature_panel",
  "rows": 58591,
  "cols": 27,
  "csv": null,
  "parquet": "/home/zilinm2/futures_anomaly_project/notebook04_post_selection_decay/features/notebook04_feature_panel.parquet"
}
{
  "name": "notebook04_strategy_pool",
  "rows": 26,
  "cols": 4,
  "csv": "/home/zilinm2/futures_anomaly_project/notebook04_post_selection_decay/tables/notebook04_strategy_pool.csv",
  "parquet": "/home/zilinm2/futures_anomaly_project/notebook04_post_selection_decay/tables/notebook04_strategy_pool.parquet"
}
{
  "name": "notebook04_symbol_candidate_strategy_returns",
  "rows": 1484852,
  "cols": 29,
  "csv": null,
  "parquet": "/home/zilinm2/futures_anomaly_project/notebook04_post_selection_decay/strategy_returns/notebook04_symbol_candidate_strategy_returns.parquet"
}
{
  "name": "notebook04_strategy_portfolio_daily_returns",
  "rows": 64626,
  "cols": 14,
  "csv": "/home/zilinm2/futures_anomaly_project/notebook04_post_selection_decay/strategy_returns/notebo